# Preparación de datos demográficos de OWID (2000–2022)

**Objetivo:** descargar seis conjuntos de datos (población, nacimientos, defunciones, defunciones de menores de cinco años, gasto en salud como % del PIB y PIB per cápita), revisar sus columnas, preparar cada uno por separado y unirlos por `Entity` y `Year`.

Este notebook genera **dos archivos**:
- `demografia_owid_2000_2022.csv`: el dataset base construido en clase (población, nacimientos, defunciones, defunciones de menores de 5 años y sus tasas).
- `demografia_gdp_health_2000_2023.csv`: el dataset final de la tarea, que agrega `percentage_health` y `gdp` al dataset base, con exactamente 8 columnas.

Ejecuta las celdas en orden. Requiere `pandas` y `matplotlib`, además de acceso a internet. Fuente: [Our World in Data](https://ourworldindata.org/).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

OPCIONES_OWID = {"User-Agent": "Our World In Data data fetch/1.0"}
BASE = "https://ourworldindata.org/grapher/"
SUFIJO = ".csv?v=1&csvType=full&useColumnShortNames=true"

## 1. Descargar los archivos

Las direcciones son cadenas de texto simples. Cada archivo se carga en su propio DataFrame.

In [ ]:
url_population = BASE + "population-with-un-projections" + SUFIJO
url_births = BASE + "number-of-births-per-year" + SUFIJO
url_deaths = BASE + "number-of-deaths-per-year" + SUFIJO
url_child_deaths = BASE + "number-of-child-deaths-unwpp" + SUFIJO
url_health = BASE + "total-healthcare-expenditure-gdp" + SUFIJO
url_gdp = BASE + "gdp-per-capita-worldbank" + SUFIJO

population = pd.read_csv(url_population, storage_options=OPCIONES_OWID)
births = pd.read_csv(url_births, storage_options=OPCIONES_OWID)
deaths = pd.read_csv(url_deaths, storage_options=OPCIONES_OWID)
child_deaths = pd.read_csv(url_child_deaths, storage_options=OPCIONES_OWID)
health = pd.read_csv(url_health, storage_options=OPCIONES_OWID)
gdp_percap = pd.read_csv(url_gdp, storage_options=OPCIONES_OWID)

print("Filas y columnas:")
print("population:", population.shape)
print("births:", births.shape)
print("deaths:", deaths.shape)
print("child_deaths:", child_deaths.shape)
print("health:", health.shape)
print("gdp_percap:", gdp_percap.shape)

## 2. Explorar las columnas

OWID puede modificar los nombres de los indicadores. Revisamos las columnas reales antes de seleccionarlas.

In [ ]:
for nombre, tabla in [("population", population), 
                      ("births", births),
                      ("deaths", deaths), 
                      ("child_deaths", child_deaths),
                      ("health", health),
                      ("gdp_percap", gdp_percap)]:
    print(f"\n{nombre}: {tabla.columns.tolist()}")
    display(tabla.head(3))

## 3. Preparar población

Elegimos la columna de **estimaciones históricas** para ambos sexos y todas las edades; excluimos las proyecciones. Después conservamos los años de interés.

In [ ]:
columnas_population = [
    columna for columna in population.columns
        if "population" in columna and "estimates" in columna]

print("Columnas candidatas:", columnas_population)

assert len(columnas_population) == 1, "Revisar manualmente el indicador de población mostrado arriba"


col_population = columnas_population[0]

population_limpio = population[["entity", "year", col_population]].copy()

population_limpio = population_limpio.rename(columns={col_population: "population"})

population_limpio = population_limpio[population_limpio["year"].between(2000, 2022)].copy()

print(population_limpio.shape)
display(population_limpio.head())

## 4. Preparar nacimientos

Seleccionamos el número anual de nacimientos históricos. Revisamos qué entidades comparte con las otras tablas.

In [ ]:
columnas_births = [c for c in births.columns
                  if "births" in c and "estimates" in c]


print("Columnas candidatas:", columnas_births)
assert len(columnas_births) == 1, "Revisar manualmente el indicador de nacimientos mostrado arriba"
col_births = columnas_births[0]

births_limpio = births[["entity", "year", col_births]].copy()
births_limpio = births_limpio.rename(columns={col_births: "births"})
births_limpio = births_limpio[births_limpio["year"].between(2000, 2022)].copy()
print(births_limpio.shape)
display(births_limpio.head())

## 5. Preparar defunciones totales

In [ ]:
columnas_deaths = [c for c in deaths.columns
                   if "deaths" in c and "estimates" in c]

print("Columnas candidatas:", columnas_deaths)
assert len(columnas_deaths) == 1, "Revisar manualmente el indicador de defunciones mostrado arriba"

col_deaths = columnas_deaths[0]

deaths_limpio = deaths[["entity", "year", col_deaths]].copy()
deaths_limpio = deaths_limpio.rename(columns={col_deaths: "deaths"})
deaths_limpio = deaths_limpio[deaths_limpio["year"].between(2000, 2022)].copy()

print(deaths_limpio.shape)
display(deaths_limpio.head())

## 6. Preparar defunciones de menores de cinco años

Este indicador corresponde a niños de 0 a 4 años; no debe confundirse con mortalidad de menores de un año.

In [ ]:
columnas_child = [c for c in child_deaths.columns
                  if c not in ["entity", "code", "year"]]

print("Columnas candidatas:", columnas_child)

assert len(columnas_child) == 1, "Revisar manualmente el indicador de menores de cinco años mostrado arriba"

col_child = columnas_child[0]

child_deaths_limpio = child_deaths[["entity", "year", col_child]].copy()
child_deaths_limpio = child_deaths_limpio.rename(columns={col_child: "child_deaths_under_5"})

child_deaths_limpio = child_deaths_limpio[child_deaths_limpio["year"].between(2000, 2022)].copy()

print(child_deaths_limpio.shape)
display(child_deaths_limpio.head())

## 7. Preparar gasto en salud (% del PIB)

Indicador único de gasto total en salud como porcentaje del PIB nacional.

In [ ]:
columnas_health = [c for c in health.columns if c not in ["entity", "code", "year"]]

if len(columnas_health) > 1:
    columnas_health = [c for c in columnas_health if "health" in c.lower()]

print("Columnas candidatas:", columnas_health)
assert len(columnas_health) == 1, "Revisar manualmente el indicador de gasto en salud mostrado arriba"

col_health = columnas_health[0]

health_limpio = health[["entity", "year", col_health]].copy()
health_limpio = health_limpio.rename(columns={col_health: "percentage_health"})
health_limpio = health_limpio[health_limpio["year"].between(2000, 2023)].copy()

print(health_limpio.shape)
display(health_limpio.head())

## 8. Preparar PIB per cápita

PIB per cápita (ajustado por inflación y costo de vida, World Bank vía OWID). También es un indicador único por entidad y año.

In [ ]:
columnas_gdp = [c for c in gdp_percap.columns if c not in ["entity", "code", "year"]]

if len(columnas_gdp) > 1:
    columnas_gdp = [c for c in columnas_gdp if "gdp" in c.lower()]

print("Columnas candidatas:", columnas_gdp)
assert len(columnas_gdp) == 1, "Revisar manualmente el indicador de PIB per cápita mostrado arriba"

col_gdp = columnas_gdp[0]

gdp_limpio = gdp_percap[["entity", "year", col_gdp]].copy()
gdp_limpio = gdp_limpio.rename(columns={col_gdp: "gdp"})
gdp_limpio = gdp_limpio[gdp_limpio["year"].between(2000, 2023)].copy()

print(gdp_limpio.shape)
display(gdp_limpio.head())

## 9. Revisar llaves y cobertura

Cada combinación `Entity`–`Year` debe aparecer una sola vez en cada tabla. `Code` no se utiliza como llave porque puede faltar en agregados regionales.

In [ ]:
tablas = [("population", population_limpio), ("births", births_limpio),
          ("deaths", deaths_limpio), ("child_deaths", child_deaths_limpio),
          ("health", health_limpio), ("gdp", gdp_limpio)]

for nombre, tabla in tablas:
    duplicados = tabla.duplicated(["entity", "year"]).sum()
    print(nombre, "| entidades:", tabla["entity"].nunique(),
          "| años:", tabla["year"].min(), "a", tabla["year"].max(),
          "| llaves duplicadas:", duplicados)
    assert duplicados == 0, f"Llaves duplicadas en {nombre}"

entidades_comunes = set(population_limpio["entity"]) & set(births_limpio["entity"]) \
                  & set(deaths_limpio["entity"]) & set(child_deaths_limpio["entity"]) \
                  & set(health_limpio["entity"]) & set(gdp_limpio["entity"])

print("Entidades presentes en las seis fuentes:", len(entidades_comunes))
print(sorted(entidades_comunes)[:30])

## 10. Unir las tablas base (población, nacimientos, defunciones, menores de 5 años)

Usamos `inner`: permanecen únicamente las combinaciones de entidad y año presentes en ambos DataFrames. `validate="one_to_one"` detecta llaves duplicadas. Este es el dataset construido en clase, antes de agregar salud y PIB.

In [ ]:
population_births = pd.merge(
    population_limpio, births_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de población + nacimientos:", population_births.shape)
display(population_births.head())

In [ ]:
population_births_deaths = pd.merge(
    population_births, deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar defunciones:", population_births_deaths.shape)
display(population_births_deaths.head())

In [ ]:
demografia = pd.merge(
    population_births_deaths, child_deaths_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Dataset base unido:", demografia.shape)
display(demografia.head())

## 11. Revisar faltantes y crear tasas (dataset base)

Eliminamos solo filas con algún indicador necesario ausente. Las tasas de nacimientos y defunciones se expresan por 1,000 habitantes. La razón de muertes de menores de cinco años usa nacimientos del **mismo año** como denominador; es una medida descriptiva aproximada, no una tasa de mortalidad de cohorte.

In [ ]:
print("Valores faltantes antes de limpiar:")
print(demografia.isna().sum())
print("Filas antes de limpiar:", len(demografia))

demografia = demografia.dropna(subset=["population", "births", "deaths", "child_deaths_under_5"]).copy()
demografia = demografia[(demografia["population"] > 0) & (demografia["births"] > 0)].copy()

print("Filas después de limpiar:", len(demografia))
print("Entidades después de limpiar:", demografia["entity"].nunique())
assert not demografia.empty, "No hay entidades y años coincidentes; revisar fuentes y columnas"

In [ ]:
demografia["birth_rate_per_1000"] = demografia["births"] / demografia["population"] * 1000
demografia["death_rate_per_1000"] = demografia["deaths"] / demografia["population"] * 1000
demografia["under5_deaths_per_1000_births_same_year"] = (
    demografia["child_deaths_under_5"] / demografia["births"] * 1000
)

demografia = demografia[[
    "entity", "year", "population", "births", "deaths", "child_deaths_under_5",
    "birth_rate_per_1000", "death_rate_per_1000",
    "under5_deaths_per_1000_births_same_year"
]].sort_values(["entity", "year"]).reset_index(drop=True)

display(demografia.head())
display(demografia.describe(include="all"))

## 12. Visualizar una entidad disponible (dataset base)

Puedes cambiar `entidad` por cualquier nombre de `demografia["entity"].unique()`.

In [ ]:
entidad = "Mexico" if "Mexico" in demografia["entity"].values else demografia["entity"].iloc[0]
ejemplo = demografia[demografia["entity"] == entidad].sort_values("year")

ejemplo.plot(x="year", y=["birth_rate_per_1000", "death_rate_per_1000"],
             marker="o", figsize=(9, 4), title=f"Tasas demográficas: {entidad}")
plt.ylabel("Por 1,000 habitantes")
plt.grid(alpha=0.3)
plt.show()

## 13. Guardar el dataset base (creado en clase)

El CSV se guarda en `data/processed`. Contiene solamente entidades y años compartidos por las cuatro fuentes originales.

In [ ]:
from pathlib import Path

carpeta_salida = Path("../data/processed")
carpeta_salida.mkdir(parents=True, exist_ok=True)

salida = carpeta_salida / "demografia_owid_2000_2022.csv"
demografia.to_csv(salida, index=False)

print(f"Archivo guardado: {salida.resolve()} | filas: {len(demografia)}")

## 14. Integrar gasto en salud y PIB per cápita

Tarea: agregamos `percentage_health` y `gdp` al dataset base creado en clase, uniendo por `entity` y `year`.

In [ ]:
demografia_completo = pd.merge(
    demografia, health_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar gasto en salud:", demografia_completo.shape)
display(demografia_completo.head())

In [ ]:
demografia_completo = pd.merge(
    demografia_completo, gdp_limpio,
    on=["entity", "year"], how="inner", validate="one_to_one"
)
print("Después de agregar PIB per cápita:", demografia_completo.shape)
display(demografia_completo.head())

## 15. Revisar faltantes y seleccionar las 8 columnas finales

El dataset final conserva exactamente las 8 columnas solicitadas: `entity`, `year`, `population`, `births`, `deaths`, `child_deaths_under_5`, `percentage_health` y `gdp`.

In [ ]:
print("Valores faltantes antes de limpiar:")
print(demografia_completo[["percentage_health", "gdp"]].isna().sum())
print("Filas antes de limpiar:", len(demografia_completo))

demografia_completo = demografia_completo.dropna(subset=["percentage_health", "gdp"]).copy()

demografia_completo = demografia_completo[[
    "entity", "year", "population", "births", "deaths", "child_deaths_under_5",
    "percentage_health", "gdp"
]].sort_values(["entity", "year"]).reset_index(drop=True)

print("Filas después de limpiar:", len(demografia_completo))
print("Columnas finales:", demografia_completo.columns.tolist())
display(demografia_completo.head())
display(demografia_completo.describe(include="all"))

## 16. Visualizar PIB y gasto en salud de una entidad

Puedes cambiar `entidad` por cualquier nombre de `demografia_completo["entity"].unique()`.

In [ ]:
entidad = "Mexico" if "Mexico" in demografia_completo["entity"].values else demografia_completo["entity"].iloc[0]
ejemplo = demografia_completo[demografia_completo["entity"] == entidad].sort_values("year")

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(ejemplo["year"], ejemplo["gdp"], marker="o", color="tab:blue", label="PIB per cápita")
ax1.set_xlabel("Año")
ax1.set_ylabel("PIB per cápita (USD)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(ejemplo["year"], ejemplo["percentage_health"], marker="s", color="tab:red", label="Gasto en salud (% PIB)")
ax2.set_ylabel("Gasto en salud (% del PIB)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title(f"PIB per cápita y gasto en salud: {entidad}")
fig.tight_layout()
plt.show()

## 17. Guardar el dataset completo

El CSV final se guarda en `data/processed`. Contiene entidad, año y las 8 columnas solicitadas por la tarea.

In [ ]:
salida_completo = carpeta_salida / "demografia_gdp_health_2000_2023.csv"
demografia_completo.to_csv(salida_completo, index=False)

print(f"Archivo guardado: {salida_completo.resolve()} | filas: {len(demografia_completo)}")